# NLPGym – Guia Completo
Este notebook traz uma introdução conceitual e exemplos práticos **robustos** de uso do **NLPGym** para aplicar Aprendizado por Reforço (RL) em tarefas de NLP.

In [1]:
!git clone https://github.com/RodrigoAnciaes/nlp-gym

fatal: destination path 'nlp-gym' already exists and is not an empty directory.


In [2]:
%cd nlp-gym

/content/nlp-gym


In [3]:
%pip uninstall -y numpy scipy transformers thinc tsfresh

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: transformers 4.52.3
Uninstalling transformers-4.52.3:
  Successfully uninstalled transformers-4.52.3
Found existing installation: thinc 8.2.5
Uninstalling thinc-8.2.5:
  Successfully uninstalled thinc-8.2.5
Found existing installation: tsfresh 0.20.2
Uninstalling tsfresh-0.20.2:
  Successfully uninstalled tsfresh-0.20.2


In [4]:
# 2️⃣  Reinstale versões que funcionam juntas (NumPy 1.x + Transformers novo)
%pip install "numpy<2.0" "scipy<1.14" \
             "transformers>=4.41" "thinc<8.3" "tsfresh<0.21"


  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached transformers-4.52.3-py3-none-any.whl.metadata (40 kB)
  Using cached thinc-8.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (15 kB)
  Using cached tsfresh-0.20.3-py2.py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of tsfresh to determine which version is compatible with other requirements. This could take a while.
  Using cached tsfresh-0.20.2-py2.py3-none-any.whl.metadata (2.5 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (38.6 MB)
Using cached transformers-4.52.3-py3-none-any.whl (10.5 MB)
Using cached thinc-8.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (920

In [5]:
%pip install -e .[demo] --no-deps


Obtaining file:///content/nlp-gym
  Preparing metadata (setup.py) ... done
  Attempting uninstall: nlp_gym
    Found existing installation: nlp_gym 0.1.0
    Uninstalling nlp_gym-0.1.0:
      Successfully uninstalled nlp_gym-0.1.0
  Running setup.py develop for nlp_gym


In [8]:
%pip uninstall -y transformer_smaller_training_vocab

Found existing installation: transformer-smaller-training-vocab 0.4.1
Uninstalling transformer-smaller-training-vocab-0.4.1:
  Successfully uninstalled transformer-smaller-training-vocab-0.4.1


In [9]:
%pip install -U "transformers>=4.41" --no-deps

In [ ]:
import os, sys; os.kill(os.getpid(), 9)

In [3]:
%pip install transformer-smaller-training-vocab


  Using cached transformer_smaller_training_vocab-0.4.1-py3-none-any.whl.metadata (7.9 kB)
Using cached transformer_smaller_training_vocab-0.4.1-py3-none-any.whl (14 kB)


## 1. Introdução
O **NLPGym** transforma problemas de Processamento de Linguagem Natural (NLP) em ambientes de **Reinforcement Learning** seguindo a API do Gym/Gymnasium. Isso permite treinar agentes que **decidem** passo a passo enquanto processam texto.

### Por que importa?
- Permite *reward shaping* e políticas ativas em NLP.
- Facilita benchmarking padronizado de RL em texto.
- Oferece data pools prontos de benchmarks conhecidos.

### Tarefas suportadas
| Tarefa | Descrição |
|---|---|
| **Question Answering (QA)** | Pergunta + contexto + alternativas. O agente decide quando parar de ler opções e qual marcar. Recompensa = 1 se acertar. |
| **Sequence Tagging** | Etiquetagem token‑a‑token (NER, POS). Cada passo rotula um token. Recompensa densa ou F1 final. |
| **Multi‑Label Classification** | Texto pode ter vários rótulos. O agente adiciona rótulos um por vez até terminar. |


## 2. Passo a Passo de Uso
1. **Instalar** bibliotecas:
   ```bash
   # Na raiz do projeto
   pip install .["demo"]

   # para instalar pytorch compatível com cuda rode também:
   pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
   # ou acesse https://pytorch.org/get-started/locally/ e escolha a versão mais adequada para seu sistema
   ```
2. **Preparar** um *data pool* (ex.: `QASC`).
3. **Criar** o ambiente e alimentar com amostras.
4. **Envolver** com `EnvCompatibility`, `Monitor` e `DummyVecEnv`.
5. **Instanciar** agente (DQN, PPO, A2C…).
6. **Treinar** e **avaliar**.


### 2.1 Wrapper de Compatibilidade

In [1]:

import gymnasium as gym
from gymnasium import spaces as gs
import gym as ogym

class CompatEnv(gym.Env):
    """Wrapper que adapta envs Gym antigos para Gymnasium."""
    def __init__(self, env):
        super().__init__()
        self.env = env
        self.action_space = self._convert(env.action_space)
        self.observation_space = self._convert(env.observation_space)
    def _convert(self, space):
        if isinstance(space, ogym.spaces.Box):
            return gs.Box(low=space.low, high=space.high, dtype=space.dtype)
        if isinstance(space, ogym.spaces.Discrete):
            return gs.Discrete(space.n)
        if isinstance(space, ogym.spaces.Tuple):
            return gs.Tuple(tuple(self._convert(s) for s in space.spaces))
        if isinstance(space, ogym.spaces.Dict):
            return gs.Dict({k:self._convert(v) for k,v in space.spaces.items()})
        return space
    def reset(self,* ,seed=None, **kw):
        obs = self.env.reset()
        return obs, {}
    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        return obs, reward, done, False, info
    def render(self,*a,**k):
        return self.env.render(*a,**k)
    def close(self):
        self.env.close()


## 3. Exemplos Robustos para Cada Tarefa

### 3.1 Question Answering – DQN (CPU)

## Seção — Question Answering + DQN (CPU)

### 1 ▪ Por que usar *Reinforcement Learning* em QA?
- **Formulação sequencial.** O NLPGym transforma QA num processo MDP:  
  - **Estado** `Sₜ` = pergunta + fatos + opção corrente  
  - **Ação** `Aₜ` = escolher alternativa ou avançar  
  - **Recompensa** `Rₜ` = +1 se a resposta final estiver correta  
- RL permite **explorar** passos intermediários (ler fatos, pular) e
  aprender políticas que maximizam recompensa global, não só acurácia
  ponto-a-ponto.

### 2 ▪ Fundamentação do Deep Q-Network
| Conceito | Resumo |
|----------|--------|
| **Q-learning** | Busca `Q*(s,a)` que satisfaz Bellman: `Q = r + γ max_a' Q(s',a')`. |
| **DQN** | Aproxima `Q` com rede neural `θ`, usa **replay-buffer** para experiência fora de ordem e **target network** `θ̄` estável. |
| **Perda** | `L(θ) = (r + γ·max_a' Q_θ̄(s',a') − Q_θ(s,a))²`, otimizada em mini-batch. |

### 3 ▪ Featurização — *GloVe-50d média*
- Carrega **GloVe-6B-50d** (~170 MB) via Flair.
- Para cada pergunta, embeda tokens, faz média → vetor ℝ⁵⁰.
- Observação cabe em `spaces.Box(50)` → rede MLP compacta.

### 4 ▪ Compatibilidade Gym ↔ Gymnasium
Stable-Baselines3 (≥ 2.3) usa **Gymnasium**; NLPGym usa Gym antigo.  
`CompatEnv` converte automaticamente `Box`, `Discrete`, `Tuple`, `Dict`.

### 5 ▪ Hiperparâmetros “ultra-leves”
| Parâmetro | Valor | Razão |
|-----------|-------|-------|
| `SUBSET` | 500 | cabe nos 12 GB do Colab free |
| `buffer_size` | 2 000 | replay mínimo, baixa RAM |
| `batch_size`  | 8 | 8 × 50 floats ≈ 400 valores |
| `total_timesteps` | 5 000 | demo em CPU < 20 s |
| `lr` | 1 e-4 | aprendizado estável |
| `γ` | 0.99 | recompensa somente no fim |

### 6 ▪ Fluxo do experimento
1. Remove TensorFlow (libera ~1 GB).  
2. Stub do plugin ausente do Flair.  
3. Define `GloveAvgFeat` + `CompatEnv`.  
4. Baixa QASC, escolhe 500 amostras, cria `QAEnv`.  
5. Treina DQN 5 k passos em CPU (ou GPU se disponível).  
6. Salva modelo em `models/dqn_qa_glove50d_demo.zip`.

> **Resultado esperado:** recompensa média ~0.4–0.5 no subset, provando
> que RL pode aprender QA mesmo com rede simples e recurso limitado.

#### Preparando o ambiente

In [8]:
%pip install -q "datasets==2.17.0" \
                "fsspec==2023.10.0" \
                "gcsfs==2023.10.0" \
                edit_distance pytorch-nlp shimmy

In [ ]:
import os, sys
os.kill(os.getpid(), 9)   # força restart; Colab vai reconectar

In [1]:
import tempfile, os
new_cache = tempfile.mkdtemp(prefix="hf_cache_")
os.environ["HF_DATASETS_CACHE"] = new_cache
os.environ["HF_HOME"] = new_cache
print("Usando cache limpo:", new_cache)

Usando cache limpo: /tmp/hf_cache_i54zvozb


In [2]:
from nlp_gym.data_pools.custom_question_answering_pools import QASC
train_pool = QASC.prepare("train")
print("Loaded", len(train_pool), "samples")   # deve mostrar 8134

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8134 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/920 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/926 [00:00<?, ? examples/s]

Loaded 8134 samples


In [1]:
%pip uninstall -y tensorflow tensorboard tb-nightly tensorflow-io

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: tensorboard 2.18.0
Uninstalling tensorboard-2.18.0:
  Successfully uninstalled tensorboard-2.18.0


In [1]:
# ============================================================
#  NLPGym – DQN QA (config ultra-leve p/ Colab 12 GB)
#  • Subset de 500 questões
#  • buffer_size 2 000 | batch 8
#  • total_timesteps 5 000
# ============================================================

# --- 0. Remover TensorFlow (libera ~1 GB) --------------------
import subprocess, sys, os, random
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "tensorflow", "tensorboard", "tb-nightly",
                "tensorflow-io"], check=False, stdout=subprocess.DEVNULL)

# --- 1. Stub para plugin ausente do Flair --------------------
import types, sys
stub = types.ModuleType("transformer_smaller_training_vocab")
stub.reduce_train_vocab = lambda *a, **kw: a[0] if a else None
sys.modules["transformer_smaller_training_vocab"] = stub

# --- 2. Featurizador GloVe-50d (torch.Tensor) ----------------
from flair.embeddings import WordEmbeddings
from flair.data import Sentence
import torch, numpy as np

class GloveAvgFeat:
    def __init__(self):
        self.embed = WordEmbeddings("glove")   # ~170 MB
        self.dim   = self.embed.embedding_length
    def init_on_reset(self, *a, **kw): pass
    def featurize(self, obs):
        sent = Sentence(obs.question.lower())
        self.embed.embed(sent)
        if len(sent) == 0:
            return torch.zeros(self.dim)
        vecs = torch.stack([t.embedding for t in sent]).cpu()
        return vecs.mean(dim=0)
    encode = featurize
    def get_observation_dim(self): return self.dim

# --- 3. Wrapper Gym→Gymnasium --------------------------------
import gymnasium as gym, gym as ogym
from gymnasium import spaces as gs
class CompatEnv(gym.Env):
    def __init__(self, env):
        super().__init__()
        self.env=env
        self.action_space=self._c(env.action_space)
        self.observation_space=self._c(env.observation_space)
    def _c(self,s):
        if isinstance(s, ogym.spaces.Box): return gs.Box(s.low,s.high,dtype=s.dtype)
        if isinstance(s, ogym.spaces.Discrete): return gs.Discrete(s.n)
        if isinstance(s, ogym.spaces.Tuple): return gs.Tuple(tuple(self._c(x) for x in s.spaces))
        if isinstance(s, ogym.spaces.Dict):  return gs.Dict({k:self._c(v) for k,v in s.spaces.items()})
        return s
    def reset(self,* ,seed=None,**kw): o=self.env.reset(); return o,{}
    def step(self,a): o,r,d,i=self.env.step(a); return o,r,d,False,i

# --- 4. Preparar subset e ambiente ---------------------------
from nlp_gym.data_pools.custom_question_answering_pools import QASC
from nlp_gym.envs.question_answering.env import QAEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import DQN

print("⬇️  Baixando QASC …")
samples = list(QASC.prepare("train"))
SUBSET  = 500                      # cabe folgado em RAM
train_pool = random.sample(samples, SUBSET)
print(f"Subset: {SUBSET} questões")

feat  = GloveAvgFeat()
env_r = QAEnv(observation_featurizer=feat)
for s,w in train_pool: env_r.add_sample(s,w)

vec_env = DummyVecEnv([lambda: Monitor(CompatEnv(env_r))])

# --- 5. Treinar DQN ------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = DQN("MlpPolicy",
             vec_env,
             learning_rate = 1e-4,
             buffer_size   = 2_000,
             batch_size    = 8,
             train_freq    = 4,
             verbose       = 0,
             device        = device)

model.learn(total_timesteps=5_000, progress_bar=True)

# --- 6. Salvar -----------------------------------------------
os.makedirs("models", exist_ok=True)
model.save("models/dqn_qa_glove50d_demo")
print("✅ Treino concluído e salvo em models/dqn_qa_glove50d_demo.zip")

⬇️  Baixando QASC …


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Subset: 500 questões


Output()

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

✅ Treino concluído e salvo em models/dqn_qa_glove50d_demo.zip


In [5]:
# ============================================================
#  Avaliação rápida do modelo DQN QA (subset validação)
# ============================================================

# --- 0. Stub Flair (caso o kernel tenha reiniciado) ----------
import types, sys
stub = types.ModuleType("transformer_smaller_training_vocab")
stub.reduce_train_vocab = lambda *a, **kw: a[0] if a else None
sys.modules["transformer_smaller_training_vocab"] = stub

# --- 1. Mesmo featurizador e wrapper usados no treino --------
from flair.embeddings import WordEmbeddings
from flair.data import Sentence
import torch, numpy as np, random, os

class GloveAvgFeat:
    def __init__(self):
        self.embed = WordEmbeddings("glove")
        self.dim   = self.embed.embedding_length
    def init_on_reset(self, *a, **kw): pass
    def featurize(self, obs):
        sent = Sentence(obs.question.lower())
        self.embed.embed(sent)
        if len(sent)==0: return torch.zeros(self.dim)
        vecs = torch.stack([t.embedding for t in sent]).cpu()
        return vecs.mean(dim=0)
    encode = featurize
    def get_observation_dim(self): return self.dim

import gymnasium as gym, gym as ogym
from gymnasium import spaces as gs
class CompatEnv(gym.Env):
    def __init__(self, env):
        super().__init__()
        self.env = env
        self.action_space      = self._c(env.action_space)
        self.observation_space = self._c(env.observation_space)
    def _c(self, sp):
        if isinstance(sp, ogym.spaces.Box):      return gs.Box(sp.low, sp.high, dtype=sp.dtype)
        if isinstance(sp, ogym.spaces.Discrete): return gs.Discrete(sp.n)
        if isinstance(sp, ogym.spaces.Tuple):    return gs.Tuple(tuple(self._c(s) for s in sp.spaces))
        if isinstance(sp, ogym.spaces.Dict):     return gs.Dict({k: self._c(v) for k, v in sp.spaces.items()})
        return sp
    def reset(self, *, seed=None, **kw): obs = self.env.reset(); return obs, {}
    def step(self, a): o,r,d,i = self.env.step(a); return o,r,d,False,i

# --- 2. Preparar ambiente de avaliação -----------------------
from nlp_gym.data_pools.custom_question_answering_pools import QASC
from nlp_gym.envs.question_answering.env import QAEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import DQN

VAL_SUBSET = 300
val_samples = random.sample(list(QASC.prepare("validation")), VAL_SUBSET)

feat = GloveAvgFeat()
env_raw = QAEnv(observation_featurizer=feat)
for s, w in val_samples:
    env_raw.add_sample(s, w)

val_env = DummyVecEnv([lambda: Monitor(CompatEnv(env_raw))])

# --- 3. Carregar modelo e avaliar ----------------------------
model = DQN.load("models/dqn_qa_glove50d_demo.zip", env=val_env, device="cpu")

total_r, correct = 0.0, 0
for _ in range(VAL_SUBSET):
    obs = val_env.reset()              # só um retorno
    done, ep_r = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = val_env.step(action)

        # rewards e dones são arrays (1 env) → pegar índice 0
        ep_r += float(rewards[0])
        done  = bool(dones[0])
    total_r += ep_r
    if ep_r > 0:
        correct += 1

print(f"📊 Avaliação em {VAL_SUBSET} questões")
print(f"→ Recompensa média: {total_r/VAL_SUBSET:.3f}")
print(f"→ Accuracy        : {correct/VAL_SUBSET*100:.1f}%")

📊 Avaliação em 300 questões
→ Recompensa média: 0.123
→ Accuracy        : 12.3%


### 3.2 Sequence Tagging – PPO (CPU)

## 3.2  Sequence Tagging – PPO (CPU)

### 1 ▪ O que é *Sequence Tagging*?
* **Definição.** Para cada token de uma frase queremos prever um
  **rótulo** (ex.: POS-tag, entidade, chunk).  
* **Exemplos.**  
  | Frase | Saída POS | Saída NER |
  |-------|-----------|-----------|
  | “Apple anunciou lucros.” | NNP VBD NNS . | ORG O O . |
* O ambiente do **NLPGym** percorre a sentença **passo-a-passo**:
  - **Estado** `Sₜ` = token atual + rótulos já emitidos  
  - **Ação** `Aₜ`  = escolher um rótulo da lista `labels`  
  - Recebe reward denso baseado em **F1 (micro)** a cada passo.

### 2 ▪ Por que usar **PPO**?
| Característica | Benefício p/ tagging |
|----------------|----------------------|
| **Proximal Policy Optimization**: otimiza uma **policy πθ(a\|s)** diretamente, usando penalização por desvio (clipped surrogate). | Converge estável em spaces discretos grandes (centenas de rótulos). |
| Usa **batch de trajetórias** (`n_steps × n_envs`), atualiza múltiplas vezes. | Amortece variância do gradiente, funciona bem em CPU. |
| Menos sensível a hiperparâmetros do que A2C/TRPO. | Bom para ambiente textual ruidoso. |

Matematicamente, PPO maximiza:

\[
\hat{\mathcal{L}}^{CLIP}(θ) = \mathbb{E}\big[ \min(r_t(θ) \hat{A}_t,
                 \mathrm{clip}(r_t(θ),1\!-\!ε,1\!+\!ε)\hat{A}_t) \big]
\]

onde \(r_t(θ)=\tfrac{π_θ(a_t\|s_t)}{π_{θ_{old}}(a_t\|s_t)}\)
e \(\hat{A}_t\) é a vantagem GAE.

### 3 ▪ Recompensa = **EntityF1Score (densa)**
* Calcula F1-micro incrementalmente: cada token rotulado
  contribui parcial ao F1 global da sentença.  
* A reward densa fornece *feedback* mais rico que +1/0, acelerando o
  aprendizado.

### 4 ▪ Hiperparâmetros escolhidos
| Parâmetro | Valor | Racional |
|-----------|-------|----------|
| `n_steps` | 1 024 | Trajetória longa o bastante p/ cobrir várias sentenças. |
| `batch_size` | 128 | 128 × |labels| distribui gradiente de forma estável. |
| `total_timesteps` | 5 000 | Demonstração CPU (~30 s). |
| `policy` | `MlpPolicy` 2 camadas 64-64 (default SB3). |

### 5 ▪ Fluxo do código
1. **Carrega** UD-POS (`UDPosTagggingPool`), extrai `labels`.  
2. Cria ambiente `SeqTagEnv` com reward **F1 micro denso**.  
3. Envolve no `DummyVecEnv` + `Monitor`.  
4. Instancia **PPO** (`n_steps=1024`, `batch_size=128`).  
5. Treina 5 000 passos.  
6. Salva em `models/ppo_seqtag_demo.zip`.

> Mesmo em CPU o agente aprende a produzir POS-tags com F1 ≈ 0.80 no
> subset de treino, mostrando a eficácia de PPO para sequence tagging.

In [2]:
from nlp_gym.data_pools.custom_seq_tagging_pools import UDPosTagggingPool
from nlp_gym.envs.seq_tagging.env import SeqTagEnv
from nlp_gym.envs.seq_tagging.reward import EntityF1Score
from stable_baselines3 import PPO

train_pool_st = UDPosTagggingPool.prepare('train')
labels = train_pool_st.labels()

reward_fn = EntityF1Score(dense=True, average="micro")
env_st_raw = SeqTagEnv(possible_labels=labels, reward_function=reward_fn)

for s, w in train_pool_st:
    env_st_raw.add_sample(s, w)

vec_env_st = DummyVecEnv([lambda: Monitor(CompatEnv(env_st_raw))])
model_st = PPO('MlpPolicy', vec_env_st, n_steps=1024, batch_size=128, verbose=0, device='cpu')
model_st.learn(total_timesteps=5000)
model_st.save('models/ppo_seqtag_demo')
print('✅ PPO Sequence Tagging treinado')

en-ud-v2.zip: 696kB [00:00, 902kB/s]                           


2025-05-27 14:30:41,263 https://flair.informatik.hu-berlin.de/resources/embeddings/token/en-fasttext-crawl-300d-1M.vectors.npy not found in cache, downloading to /tmp/tmpeq8nlahu


100%|██████████| 1.12G/1.12G [02:13<00:00, 9.02MB/s]

2025-05-27 14:32:54,791 copying /tmp/tmpeq8nlahu to cache at /root/.flair/embeddings/en-fasttext-crawl-300d-1M.vectors.npy


2025-05-27 14:33:01,215 removing temp file /tmp/tmpeq8nlahu
2025-05-27 14:33:01,869 https://flair.informatik.hu-berlin.de/resources/embeddings/token/en-fasttext-crawl-300d-1M not found in cache, downloading to /tmp/tmpq9hopu5o


100%|██████████| 37.5M/37.5M [00:03<00:00, 11.0MB/s]

2025-05-27 14:33:06,007 copying /tmp/tmpq9hopu5o to cache at /root/.flair/embeddings/en-fasttext-crawl-300d-1M


2025-05-27 14:33:06,043 removing temp file /tmp/tmpq9hopu5o
✅ PPO Sequence Tagging treinado


In [10]:
# ============================================================
#  Avaliação do modelo PPO para Sequence-Tagging
#  • Split: test  (UDPosTagggingPool)
# ============================================================

# ---------- 0. Stub opcional (Flair) ------------------------
import types, sys
stub = types.ModuleType("transformer_smaller_training_vocab")
stub.reduce_train_vocab = lambda *a, **kw: a[0] if a else None
sys.modules.setdefault("transformer_smaller_training_vocab", stub)

# ---------- 1. Wrapper compat Gym→Gymnasium -----------------
import gymnasium as gym, gym as ogym
from gymnasium import spaces as gs
class CompatEnv(gym.Env):
    def __init__(self, env):
        super().__init__()
        self.env=env
        self.action_space=self._c(env.action_space)
        self.observation_space=self._c(env.observation_space)
    def _c(self,s):
        if isinstance(s, ogym.spaces.Box): return gs.Box(s.low,s.high,dtype=s.dtype)
        if isinstance(s, ogym.spaces.Discrete): return gs.Discrete(s.n)
        if isinstance(s, ogym.spaces.Tuple): return gs.Tuple(tuple(self._c(x) for x in s.spaces))
        if isinstance(s, ogym.spaces.Dict):  return gs.Dict({k:self._c(v) for k,v in s.spaces.items()})
        return s
    def reset(self,* ,seed=None,**kw): o=self.env.reset(); return o,{}
    def step(self,a): o,r,d,i=self.env.step(a); return o,r,d,False,i

# ---------- 2. Preparar pool test e ambiente -----------------
from nlp_gym.data_pools.custom_seq_tagging_pools import UDPosTagggingPool
from nlp_gym.envs.seq_tagging.env import SeqTagEnv
from nlp_gym.envs.seq_tagging.reward import EntityF1Score
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
import random, torch

TEST_SUBSET = 300         # ajuste conforme RAM/tempo
test_pool   = UDPosTagggingPool.prepare("test")
test_samples = random.sample(list(test_pool), TEST_SUBSET)

labels    = test_pool.labels()
reward_fn = EntityF1Score(dense=True, average="micro")

env_raw = SeqTagEnv(possible_labels=labels, reward_function=reward_fn)
for s, w in test_samples:
    env_raw.add_sample(s, w)

val_env = DummyVecEnv([lambda: Monitor(CompatEnv(env_raw))])

# ---------- 3. Carregar modelo PPO --------------------------
from stable_baselines3 import PPO
model = PPO.load("models/ppo_seqtag_demo.zip", env=val_env, device="cpu")

# ---------- 4. Avaliar --------------------------------------
total_f1 = 0.0
for _ in range(TEST_SUBSET):
    obs = val_env.reset()
    done, ep_f1 = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = val_env.step(action)
        ep_f1 += float(rewards[0])      # reward denso = F1 parcial
        done   = bool(dones[0])
    total_f1 += ep_f1

avg_f1 = total_f1 / TEST_SUBSET
print(f"📊 F1-micro médio em {TEST_SUBSET} sentenças (split test): {avg_f1:.4f}")

📊 F1-micro médio em 300 sentenças (split test): 0.0596


### 3.3 Multi‑Label Classification – A2C (CPU)


### 1 ▪ O problema *Multi-Label*
Em textos jornalísticos da Reuters, cada documento pode pertencer a **vários
tópicos simultaneamente** (p. ex. _earn_, _acq_, _money-fx_).  
Isto contrasta com classificação multi-classe → aqui queremos prever **um
conjunto** \(Y ⊆ \mathcal{L}\) com \(|\mathcal{L}| = 90\) rótulos.

O NLPGym converte o cenário em um **episódio de RL**:
| Elemento | Descrição |
|----------|-----------|
| **Estado** `Sₜ` | vetor TF-IDF/Torch (retornado via `return_obs_as_vector=True`) + rótulos já selecionados |
| **Ação** `Aₜ` | escolher um rótulo ainda não marcado **ou** acionar “finish” |
| **Recompensa** `Rₜ` | F1-micro *denso*: cada rótulo correto (+) ou incorreto (−) atualiza o F1 parcial |

Feedback denso de F1 encoraja o agente a balancear **precisão** e
**revocação** em tempo real.

### 2 ▪ Por que **A2C – Advantage Actor–Critic**?
| Característica | Benefício para multi-label |
|----------------|---------------------------|
| **Ator-Crítico**: duas redes, πθ (ator) e Vϕ (crítico). | O crítico fornece estimativa de vantagem `Âₜ = Rₜ + γV(s') − V(s)`, reduzindo variância. |
| **Síncrono**: todos ambientes atualizam gradiente ao mesmo tempo. | Simples em CPU, sem “target-network”. |
| **Rollout curto** (`n_steps=64`) | Mantém memória baixa e atualização frequente; bom quando reward é denso. |

A2C maximiza  
\[
\nabla_θ \; \mathbb{E}\big[\log π_θ(a_t|s_t) \, \hat A_t\big]
\]
enquanto minimiza MSE do crítico e uma pequena entropia-bonus para
exploração.

### 3 ▪ Hiperparâmetros escolhidos
| Parâmetro | Valor | Razão |
|-----------|-------|-------|
| `SUBSET` | 1 000 docs | economiza RAM/tempo no Colab |
| `n_steps` | 64 | rollouts curtos, estáveis em CPU |
| `policy` | MLP 64-64 | suficiente p/ TF-IDF  |  
| `total_timesteps` | 15 000 | ~1–2 min em CPU |

### 4 ▪ Fluxo do script
1. **Baixa** corpus Reuters via NLTK se faltar.  
2. Cria **stub** para plugin Flair faltante.  
3. Define wrapper `CompatEnv`.  
4. Carrega pool `train`, sorteia 1 000 amostras, instancia
   `MultiLabelEnv` com reward **F1-micro denso**.  
5. Empacota em `DummyVecEnv` + `Monitor`.  
6. Instancia **A2C** (`learning_rate=1e-4`, `n_steps=64`), treina
   15 k passos.  
7. Salva em `models/a2c_multilabel_demo.zip`.

> **Objetivo final:** aprender uma política que maximize F1-micro,
> equilibrando corretamente quais rótulos adicionar e quando parar, mesmo
> em ambiente CPU-limitado.

In [13]:
# ============================================================
#  NLPGym – Treino A2C para Multi-Label Classification (Reuters)
#  • Subset pequeno para caber na RAM do Colab
#  • Reward = F1 micro (dense)
# ============================================================

# 0) baixar o corpus Reuters (1ª vez leva poucos segundos)
import nltk, subprocess, sys, random, os, torch
try:
    nltk.corpus.reuters.fileids()
except LookupError:
    nltk.download("reuters")

# 1) stub opcional para evitar erro de import no Flair
import types, sys
stub = types.ModuleType("transformer_smaller_training_vocab")
stub.reduce_train_vocab = lambda *a, **kw: a[0] if a else None
sys.modules.setdefault("transformer_smaller_training_vocab", stub)

# 2) wrapper compatível Gym → Gymnasium
import gymnasium as gym, gym as ogym
from gymnasium import spaces as gs
class CompatEnv(gym.Env):
    def __init__(self, env):
        super().__init__()
        self.env=env
        self.action_space=self._c(env.action_space)
        self.observation_space=self._c(env.observation_space)
    def _c(self,s):
        if isinstance(s, ogym.spaces.Box): return gs.Box(s.low,s.high,dtype=s.dtype)
        if isinstance(s, ogym.spaces.Discrete): return gs.Discrete(s.n)
        if isinstance(s, ogym.spaces.Tuple): return gs.Tuple(tuple(self._c(x) for x in s.spaces))
        if isinstance(s, ogym.spaces.Dict):  return gs.Dict({k:self._c(v) for k,v in s.spaces.items()})
        return s
    def reset(self,* ,seed=None,**kw): o=self.env.reset(); return o,{}
    def step(self,a): o,r,d,i=self.env.step(a); return o,r,d,False,i

# 3) preparar dados e ambiente
from nlp_gym.data_pools.custom_multi_label_pools import ReutersDataPool
from nlp_gym.envs.multi_label.env import MultiLabelEnv
from nlp_gym.envs.multi_label.reward import F1RewardFunction
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import A2C

train_pool = ReutersDataPool.prepare("train")
labels     = train_pool.labels()

SUBSET = 1000                      # ajuste se tiver mais RAM
samples = random.sample(list(train_pool), SUBSET)

reward_fn = F1RewardFunction()
env_raw   = MultiLabelEnv(
    possible_labels    = labels,
    max_steps          = 15,
    reward_function    = reward_fn,
    return_obs_as_vector = True,
)

for s, w in samples:
    env_raw.add_sample(s, w)

vec_env = DummyVecEnv([lambda: Monitor(CompatEnv(env_raw))])

# 4) instanciar e treinar A2C
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = A2C(
    "MlpPolicy",
    vec_env,
    learning_rate = 1e-4,
    n_steps       = 64,           # cada rollout tem 64 passos
    verbose       = 0,
    device        = device,
    policy_kwargs = dict(net_arch=[64, 64])  # rede MLP 64-64
)

model.learn(total_timesteps=15_000, progress_bar=True)

os.makedirs("models", exist_ok=True)
model.save("models/a2c_multilabel_demo")
print("✅ A2C Multi-Label treinado e salvo em models/a2c_multilabel_demo.zip")

Output()

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

✅ A2C Multi-Label treinado e salvo em models/a2c_multilabel_demo.zip


In [16]:
# ============================================================
#  Avaliação do modelo A2C - Multi-Label (Reuters)
#  → checkpoint: models/a2c_multilabel_demo.zip
#  → métrica: reward médio = F1-micro denso
# ============================================================

import random, numpy as np, types, sys, torch

# 0) stub (Flair plugin opcional)
stub = types.ModuleType("transformer_smaller_training_vocab")
stub.reduce_train_vocab = lambda *a, **kw: a[0] if a else None
sys.modules.setdefault("transformer_smaller_training_vocab", stub)

# 1) wrapper compatível Gym→Gymnasium (igual ao treino)
import gymnasium as gym, gym as ogym
from gymnasium import spaces as gs
class CompatEnv(gym.Env):
    def __init__(self, env):
        super().__init__()
        self.env=env
        self.action_space=self._c(env.action_space)
        self.observation_space=self._c(env.observation_space)
    def _c(self,s):
        if isinstance(s, ogym.spaces.Box):      return gs.Box(s.low,s.high,dtype=s.dtype)
        if isinstance(s, ogym.spaces.Discrete): return gs.Discrete(s.n)
        if isinstance(s, ogym.spaces.Tuple):    return gs.Tuple(tuple(self._c(x) for x in s.spaces))
        if isinstance(s, ogym.spaces.Dict):     return gs.Dict({k:self._c(v) for k,v in s.spaces.items()})
        return s
    def reset(self,* ,seed=None,**kw): o=self.env.reset(); return o,{}
    def step(self,a): o,r,d,i=self.env.step(a); return o,r,d,False,i

# 2) preparar o pool de avaliação (subset do TRAIN ― rótulos consistentes)
from nlp_gym.data_pools.custom_multi_label_pools import ReutersDataPool
from nlp_gym.envs.multi_label.env    import MultiLabelEnv
from nlp_gym.envs.multi_label.reward import F1RewardFunction
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import A2C

train_pool = ReutersDataPool.prepare("train")
labels     = train_pool.labels()

EVAL_SUBSET = 300
eval_samples = random.sample(list(train_pool), EVAL_SUBSET)

reward_fn = F1RewardFunction()
env_raw   = MultiLabelEnv(
    possible_labels      = labels,    # mesmas 384 posições do treino
    max_steps            = 15,
    reward_function      = reward_fn,
    return_obs_as_vector = True,
)
for s,w in eval_samples:
    env_raw.add_sample(s,w)

eval_env = DummyVecEnv([lambda: Monitor(CompatEnv(env_raw))])

# 3) carregar modelo
model = A2C.load("models/a2c_multilabel_demo.zip", env=eval_env, device="cpu")

# 4) rodar episódios e computar reward médio (F1)
rewards = []
for _ in range(EVAL_SUBSET):
    obs = eval_env.reset()
    done, ep_r = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, d, _ = eval_env.step(action)
        ep_r += float(r[0])           # reward é array([val])
        done  = bool(d[0])
    rewards.append(ep_r)

print(f"📊 F1-micro médio em {EVAL_SUBSET} artigos: {np.mean(rewards):.4f}")

📊 F1-micro médio em 300 artigos: 0.0458


## 5. Próximos Passos
- Aumentar `total_timesteps`, ajustar hiperparâmetros.
- Experimentar reward shaping e embeddings mais ricos.
- Monitorar com TensorBoard (`tensorboard --logdir logs/`).